# 🛣️ The Data Journey: From Raw Chaos to Structured Insights

Notebook ini mendokumentasikan transformasi data dalam proyek **RDV-Taxi**. Kita akan melihat bagaimana data mentah yang berantakan diolah menjadi informasi yang berkualitas tinggi.

---

In [1]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

con = duckdb.connect("../data/final/tlc.duckdb", read_only=True)
print("✅ Pipeline Auditor Ready")

NameError: name 'con' is not defined

## 🛑 Tahap 1: Kondisi Awal (Raw Data)
**Masalah Utama:** Data masih dalam format mentah, banyak kolom teknis yang tidak perlu, dan ada ribuan data sampah (anomali).

In [ ]:
raw_info = con.execute("SELECT * FROM tlc_raw LIMIT 5000").df()
print(f"📦 Total Data Mentah: {con.execute('SELECT COUNT(*) FROM tlc_raw').fetchone()[0]:,} baris")
display(raw_info.head())

print("\n⚠️ Temuan Masalah:")
print(f"- Ada {con.execute('SELECT COUNT(*) FROM tlc_raw WHERE fare_amount < 0').fetchone()[0]:,} baris dengan tarif NEGATIF.")
print(f"- Ada {con.execute('SELECT COUNT(*) FROM tlc_raw WHERE passenger_count = 0').fetchone()[0]:,} baris dengan penumpang 0.")

## 🧹 Tahap 2: Pembersihan & Engineering (Transformation Stage)
**Tindakan Engineering:**
1. **Data Type Casting:** Mengubah string/object menjadi `TIMESTAMP`, `INTEGER`, dan `FLOAT`.
2. **Filtering:** Membuang data tarif negatif dan durasi yang tidak masuk akal.
3. **Feature Engineering:** Membuat kolom baru `trip_duration_min` (durasi menit) dan `tip_percentage` yang tidak ada di data mentah.

In [ ]:
cleaned_sample = con.execute("SELECT trip_duration_min, fare_amount, tip_percentage FROM tlc_cleaned LIMIT 5").df()
print("✨ Hasil Transformasi (New Features Added):")
display(cleaned_sample)

print(f"\n✅ Data Bersih Tersisa: {con.execute('SELECT COUNT(*) FROM tlc_cleaned').fetchone()[0]:,} baris")

## 💉 Tahap 3: Pengayaan Data (Enrichment Stage)
**Masalah:** Data taksi tidak punya informasi lokasi (hanya kode angka) dan tidak ada info cuaca.

**Tindakan Engineering:**
- **Spatial Join:** Menghubungkan `LocationID` dengan koordinat GPS (Latitude/Longitude).
- **Temporal Join:** Menghubungkan waktu penjemputan dengan data cuaca historis.

In [ ]:
enrich_df = con.execute("""
    SELECT borough, weather_description, temperature_2m, pickup_latitude
    FROM fact_trips 
    WHERE weather_description != 'Unknown'
    LIMIT 5
"""
).df()
print("💉 Data Setelah Disuntik Info Lokasi & Cuaca:")
display(enrich_df)

## 🏆 Tahap 4: Hasil Akhir (Star Schema)
**Tindakan Engineering:** Mengorganisir data ke dalam bentuk **Fact Table** dan **Dimension Tables** agar kueri untuk dashboard menjadi sangat cepat.

In [ ]:
tables = con.execute("SHOW TABLES").df()
print("🏛️ Struktur Gudang Data (Data Warehouse) Selesai:")
display(tables)

print("\n🚀 READY FOR VISUALIZATION!")

In [ ]:
con.close()